# 02 — Data Quality dan Cleaningggg

Melanjutkan [Milestone 1](01_data_profiling.ipynb) untuk dataset `IndonesiaAI/cleaned-data-split-0`. Notebook memakai DuckDB untuk profiling dan Polars LazyFrame untuk transformasi. Data mentah tidak diubah.

In [1]:
from pathlib import Path

import duckdb
import polars as pl
import pyarrow.parquet as pq

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'data').exists() else cwd.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'data'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

valid_files, invalid_files = [], []
for file in sorted(RAW_DIR.glob('*.parquet')):
    try:
        pq.ParquetFile(file).schema_arrow
        valid_files.append(file)
    except Exception as error:
        invalid_files.append({'file': file.name, 'error': str(error).splitlines()[0]})

if not valid_files:
    raise RuntimeError(f'Tidak ada shard Parquet valid di {RAW_DIR}')

print(f'Shard valid: {len(valid_files)}')
if invalid_files:
    print('Shard dikecualikan (unduh ulang sebelum hasil final):')
    for issue in invalid_files:
        print(f"- {issue['file']}: {issue['error']}")

lazy_raw = pl.scan_parquet([str(file) for file in valid_files])
RAW_COLUMNS = lazy_raw.collect_schema().names()
jumlah_baris_awal = lazy_raw.select(pl.len()).collect().item()
print(f'Ukuran awal: {jumlah_baris_awal:,} baris x {len(RAW_COLUMNS)} kolom')

Shard valid: 5
Shard dikecualikan (unduh ulang sebelum hasil final):
- train-00000-of-00006-042e4a2154d9270b.parquet: Couldn't deserialize thrift: TProtocolException: Invalid data
Ukuran awal: 875,214 baris x 6 kolom


## 1. Profiling dengan DuckDB `SUMMARIZE`

DuckDB membaca Parquet langsung dari disk sehingga tidak perlu memuat seluruh dataset ke DataFrame terlebih dahulu.

In [2]:
connection = duckdb.connect()
try:
    profil = connection.execute(
        'SUMMARIZE SELECT * FROM read_parquet(?)',
        [[str(file) for file in valid_files]],
    ).to_arrow_table()
    profil = pl.from_arrow(profil)
finally:
    connection.close()

profil

C:\Users\Workshop\AppData\Local\Temp\ipykernel_3440\298437377.py:6: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  ).to_arrow_table()


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""qid""","""BIGINT""","""1""","""74661602""",229684,"""2798205.048132228""","""9044724.943825755""","""26869""","""105173""","""455979""",875214,0.00
"""question""","""VARCHAR""","""!!! I found an interesting sol…","""﻿ What would a political syste…",325828,null,null,null,null,null,875214,0.00
"""date""","""VARCHAR""","""2008/08/01""","""2023/03/04""",5235,null,null,null,null,null,875214,0.00
"""metadata""","""VARCHAR[]""","""['https://3dprinting.meta.stac…","""['https://writers.stackexchang…",322036,null,null,null,null,null,875214,0.00
"""response_j""","""VARCHAR""","""""","""ﷲ = الله ====""",526545,null,null,null,null,null,875214,0.00
"""response_k""","""VARCHAR""","""! Just as well that I have a…","""Watch only produces notificat…",589192,null,null,null,null,null,875214,0.00


## 2. Missing values dan teks kosong

Milestone 1 menunjukkan tidak ada `null`. Untuk kolom teks, nilai kosong/whitespace juga diperiksa karena secara makna merupakan missing value.

In [3]:
null_summary = (
    lazy_raw.select(pl.all().null_count()).collect()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns((pl.col('jumlah_null') / jumlah_baris_awal * 100).round(4).alias('persen_null'))
    .sort('jumlah_null', descending=True)
)

empty_text_summary = lazy_raw.select([
    pl.col(column).str.strip_chars().eq('').sum().alias(column)
    for column in ['question', 'date', 'response_j', 'response_k']
]).collect()

null_summary, empty_text_summary

(shape: (6, 3)
 ┌────────────┬─────────────┬─────────────┐
 │ kolom      ┆ jumlah_null ┆ persen_null │
 │ ---        ┆ ---         ┆ ---         │
 │ str        ┆ u32         ┆ f64         │
 ╞════════════╪═════════════╪═════════════╡
 │ qid        ┆ 0           ┆ 0.0         │
 │ question   ┆ 0           ┆ 0.0         │
 │ date       ┆ 0           ┆ 0.0         │
 │ metadata   ┆ 0           ┆ 0.0         │
 │ response_j ┆ 0           ┆ 0.0         │
 │ response_k ┆ 0           ┆ 0.0         │
 └────────────┴─────────────┴─────────────┘,
 shape: (1, 4)
 ┌──────────┬──────┬────────────┬────────────┐
 │ question ┆ date ┆ response_j ┆ response_k │
 │ ---      ┆ ---  ┆ ---        ┆ ---        │
 │ u32      ┆ u32  ┆ u32        ┆ u32        │
 ╞══════════╪══════╪════════════╪════════════╡
 │ 0        ┆ 0    ┆ 2          ┆ 0          │
 └──────────┴──────┴────────────┴────────────┘)

## 3. Penanganan missing values

Tidak ada nilai `null` untuk diimputasi. Baris dengan `question`, `date`, atau respons yang kosong dibuang karena tidak dapat membentuk contoh pertanyaan–respons yang utuh. `metadata` dibiarkan apa adanya karena merupakan daftar URL/konteks, bukan kolom kategorik untuk diimputasi.

In [4]:
TEXT_REQUIRED = ['question', 'date', 'response_j', 'response_k']

df_missing_handled = lazy_raw.filter(
    pl.all_horizontal([
        pl.col(column).is_not_null() & pl.col(column).str.strip_chars().ne('')
        for column in TEXT_REQUIRED
    ])
)

jumlah_setelah_missing = df_missing_handled.select(pl.len()).collect().item()
print(f'Baris sebelum penanganan missing: {jumlah_baris_awal:,}')
print(f'Baris sesudah penanganan missing: {jumlah_setelah_missing:,}')
print(f'Baris dihapus: {jumlah_baris_awal - jumlah_setelah_missing:,}')

Baris sebelum penanganan missing: 875,214
Baris sesudah penanganan missing: 875,212
Baris dihapus: 2


## 4. Deteksi outlier panjang pertanyaan

`qid` adalah identifier, sehingga IQR tidak diterapkan pada kolom tersebut. IQR digunakan untuk panjang `question`. Pertanyaan panjang tidak dihapus atau dipotong karena dapat berisi konteks yang valid; outlier hanya diberi penanda untuk analisis berikutnya.

In [5]:
# Kuartil dihitung DuckDB langsung dari Parquet agar tidak mematerialisasi
# kolom teks besar di memori; penandaan outlier tetap dilakukan oleh Polars.
connection = duckdb.connect()
try:
    q1, q3 = connection.execute(
        "SELECT quantile_cont(char_length(question), 0.25), quantile_cont(char_length(question), 0.75) FROM read_parquet(?) WHERE trim(response_j) <> ''",
        [[str(file) for file in valid_files]],
    ).fetchone()
finally:
    connection.close()

panjang_question = pl.col('question').str.len_chars()
iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

df_outlier_handled = df_missing_handled.with_columns([
    panjang_question.alias('panjang_question'),
    (panjang_question > batas_atas).alias('is_question_extreme'),
])

outlier_summary = df_outlier_handled.select([
    pl.len().alias('jumlah_baris'),
    pl.col('is_question_extreme').sum().alias('question_ekstrem'),
    pl.col('panjang_question').max().alias('panjang_maksimum'),
]).collect()

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas IQR: {batas_bawah} s.d. {batas_atas}')
outlier_summary

Q1=313.0, Q3=919.0, IQR=606.0
Batas IQR: 0 s.d. 1828.0


jumlah_baris,question_ekstrem,panjang_maksimum
u32,u32,u32
875212,57118,33832


## 5. Deduplikasi

`qid` berulang pada dataset ini, sehingga bukan kunci bisnis yang aman. Penghapusan duplikat hanya dilakukan pada baris yang identik pada keenam kolom asli. Kolom penanda outlier tetap dipertahankan pada hasil bersih.

In [6]:
# Materialisasikan hasil cleaning sekali agar transformasi sumber tidak diulang.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGING_PATH = OUTPUT_PATH.with_name('_dataset_sebelum_deduplikasi.parquet')
DEDUP_DIR = OUTPUT_PATH.with_name('_deduplikasi_parts')
DEDUP_PARTITIONS = 64
DEDUP_DIR.mkdir(parents=True, exist_ok=True)
df_outlier_handled.sink_parquet(STAGING_PATH)

# Baris identik memiliki qid yang sama, sehingga masuk ke partisi yang sama.
# DISTINCT per partisi membatasi memori yang dibutuhkan setiap proses.
connection = duckdb.connect()
try:
    jumlah_setelah_missing = connection.execute(
        'SELECT count(*) FROM read_parquet(?)', [str(STAGING_PATH)]
    ).fetchone()[0]
    partition_sql_path = str(DEDUP_DIR / "raw").replace("'", "''")
    connection.execute(
        f"COPY (SELECT *, (qid % {DEDUP_PARTITIONS}) AS _bucket "
        f"FROM read_parquet(?)) TO '{partition_sql_path}' "
        "(FORMAT PARQUET, PARTITION_BY (_bucket))",
        [str(STAGING_PATH)],
    )
    for bucket in range(DEDUP_PARTITIONS):
        source_part = DEDUP_DIR / f"raw\\_bucket={bucket}" / "data_0.parquet"
        if not source_part.exists():
            continue
        result_part = DEDUP_DIR / f"part-{bucket:03d}.parquet"
        if result_part.exists():
            result_part.unlink()
        result_sql_path = str(result_part).replace("'", "''")
        connection.execute(
            f"COPY (SELECT DISTINCT * EXCLUDE (_bucket) "
            f"FROM read_parquet('{str(source_part).replace(chr(39), chr(39) * 2)}')) "
            f"TO '{result_sql_path}' (FORMAT PARQUET)"
        )
    jumlah_baris_akhir = connection.execute(
        'SELECT count(*) FROM read_parquet(?)', [str(DEDUP_DIR / 'part-*.parquet')]
    ).fetchone()[0]
finally:
    connection.close()

# Gabungkan kembali menjadi satu file Parquet untuk tahap berikutnya.
df_dedup = pl.scan_parquet(str(DEDUP_DIR / 'part-*.parquet'))
df_dedup.sink_parquet(OUTPUT_PATH)
df_dedup = pl.scan_parquet(OUTPUT_PATH)

print(f'Baris sebelum deduplikasi: {jumlah_setelah_missing}')
print(f'Baris sesudah deduplikasi: {jumlah_baris_akhir}')
print(f"Duplikat identik dihapus: {jumlah_setelah_missing - jumlah_baris_akhir}")

Baris sebelum deduplikasi: 875212
Baris sesudah deduplikasi: 875212
Duplikat identik dihapus: 0


## 6. Validasi data dasar

Aturan validasi memastikan tidak ada missing/teks kosong, tanggal dapat diurai, metadata tersedia, dan penanda outlier konsisten dengan panjang pertanyaan.

In [7]:
validasi = df_dedup.with_columns(
    pl.col('date').str.to_date(strict=False).alias('date_parsed')
).select([
    pl.len().alias('total_baris'),
    pl.any_horizontal([pl.col(column).is_null() for column in RAW_COLUMNS]).sum().alias('baris_dengan_null'),
    pl.any_horizontal([pl.col(column).str.strip_chars().eq('') for column in TEXT_REQUIRED]).sum().alias('baris_dengan_teks_kosong'),
    pl.col('date_parsed').is_null().sum().alias('tanggal_tidak_valid'),
    pl.col('metadata').list.len().eq(0).sum().alias('metadata_kosong'),
    (pl.col('is_question_extreme') != (pl.col('panjang_question') > batas_atas)).sum().alias('flag_outlier_tidak_konsisten'),
]).collect(engine='streaming')
validasi

total_baris,baris_dengan_null,baris_dengan_teks_kosong,tanggal_tidak_valid,metadata_kosong,flag_outlier_tidak_konsisten
u32,u32,u32,u32,u32,u32
875212,0,0,0,0,0


## 7. Hasil data bersih

Hasil deduplikasi sudah ditulis langsung ke Parquet oleh DuckDB pada sel sebelumnya.

In [8]:
print(f'Baris sebelum cleaning: {jumlah_baris_awal:,}')
print(f'Baris sesudah cleaning: {jumlah_baris_akhir:,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 875,214
Baris sesudah cleaning: 875,212
Data bersih tersimpan di: D:\semester 7\tugas1-Luqmanul-Hakim-141-2026\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

- Proses cleaning mengolah lima shard Parquet valid dengan total **875.214 baris**.
- Dataset pada shard valid tidak memiliki nilai `null`; dua respons kosong ditangani sebagai missing agar data yang dianalisis tetap bermakna.
- Panjang pertanyaan dianalisis menggunakan metode IQR. Sebanyak **57.118** pertanyaan ekstrem diberi penanda `is_question_extreme` agar tetap tersedia untuk analisis lanjutan.
- Deduplikasi mencocokkan keenam kolom asli untuk mempertahankan satu salinan dari setiap baris yang identik, sementara `qid` tetap dipertahankan sebagai atribut data.
- Validasi akhir mencakup kelengkapan data, format tanggal, metadata, dan konsistensi penanda outlier. Hasil bersih disimpan dalam format Parquet untuk penggunaan berikutnya.